# Module 5 Lab — ML Algorithms (extended practice set)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/applied-data-science-python/blob/main/module-05-machine-learning/notebooks/Module05_ML_Algorithms_Lab.ipynb)

**Applied Data Science with Python** · Open Educational Resource · CC BY 4.0

Each section follows the same path: **Learning Objectives → Concept → Guided Practice → Your Turn**. Run the cells in order. In *Your Turn* cells, replace every `___` (or the `# YOUR CODE HERE` line) and run the cell; the `assert` checks tell you when you are right.

Estimated time: 8 hours of practical work.

In [ ]:
# Run this cell first. It loads course datasets from the repository
# (works locally, in JupyterLab, and in Google Colab).
import os
import pandas as pd

REPO_RAW = "https://raw.githubusercontent.com/YOUR-GITHUB-USERNAME/applied-data-science-python/main/datasets/"   # <- replace YOUR-GITHUB-USERNAME once you publish the repo

def load_data(name, **kwargs):
    """Load a course dataset by file name, e.g. load_data("titanic.csv")."""
    for folder in ("../../datasets", "../datasets", "datasets", "."):
        path = os.path.join(folder, name)
        if os.path.exists(path):
            return pd.read_csv(path, **kwargs)
    return pd.read_csv(REPO_RAW + name, **kwargs)

## Part 1 — The ML pipeline and the train/test split

### Learning Objectives
- Prepare features `X` and a target `y`
- Split data reproducibly
- Set a baseline before modelling

### Concept
Fit on the **training** set; evaluate once on the **test** set. A baseline (for example, always predicting the most common class) tells you whether a model is adding value.

### Guided Practice

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             mean_squared_error, r2_score, ConfusionMatrixDisplay)
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans

titanic = load_data("titanic.csv")
titanic["age"] = titanic["age"].fillna(titanic.groupby("pclass")["age"].transform("median"))
titanic["embarked"] = titanic["embarked"].fillna("S")
titanic["is_female"] = (titanic["sex"] == "female").astype(int)
titanic["family_size"] = titanic["sibsp"] + titanic["parch"] + 1
titanic = pd.get_dummies(titanic, columns=["embarked"], prefix="emb", dtype=int)
titanic.head()

### Your Turn
**Exercise 1.1** — Build `X` from `pclass, is_female, age, fare, family_size, emb_C, emb_Q, emb_S` and `y` = `survived`. Split 80/20 with `random_state=42` and `stratify=y`. Compute the majority-class `baseline` accuracy on the test set.

In [ ]:
features = ["pclass", "is_female", "age", "fare", "family_size", "emb_C", "emb_Q", "emb_S"]
X = ___
y = ___
X_train, X_test, y_train, y_test = ___
baseline = ___

assert len(X_test) == 179 and abs(y_train.mean() - y_test.mean()) < 0.01
print(f"Baseline accuracy: {baseline:.3f}")

## Part 2 — Classification

### Learning Objectives
- Train a logistic regression classifier
- Evaluate it with accuracy, a confusion matrix and a classification report

### Concept
`LogisticRegression` outputs probabilities via the sigmoid function; `predict()` applies a 0.5 threshold. Precision = of predicted positives, how many are right; recall = of actual positives, how many we caught.

### Your Turn
**Exercise 2.1** — Train `clf` (use `max_iter=1000`), predict on the test set (`pred`), and compute `acc`. Beat the baseline!

In [ ]:
clf = ___
pred = ___
acc = ___

assert acc > baseline + 0.1
print(f"Accuracy {acc:.3f} vs baseline {baseline:.3f}")

**Exercise 2.2** — Display the confusion matrix (`ConfusionMatrixDisplay.from_predictions`) and print the classification report. In a comment: which error is more common, missing survivors or false alarms?

In [ ]:
# YOUR CODE HERE

**Exercise 2.3 (threshold)** — Use `predict_proba` to get survival probabilities. Predict 'survived' when the probability is above **0.35** instead of 0.5 (`pred_35`). How do recall and accuracy change?

In [ ]:
proba = ___
pred_35 = ___

from sklearn.metrics import recall_score
assert recall_score(y_test, pred_35) >= recall_score(y_test, pred)
print("recall 0.5:", round(recall_score(y_test, pred), 3), "| recall 0.35:", round(recall_score(y_test, pred_35), 3))

## Part 3 — Regression

### Learning Objectives
- Train a linear regression model
- Evaluate with MSE, RMSE and R²

### Concept
RMSE is in the same units as the target, so it is the easiest error to explain. R² is the share of variation explained.

### Your Turn
**Exercise 3.1** — Predict `median_house_value` from `median_income, housing_median_age, total_rooms, population, households, latitude, longitude` plus one-hot `ocean_proximity`. Drop rows with missing values first. Report `rmse` and `r2` on a 20% test set (`random_state=42`).

In [ ]:
housing = load_data("california_housing.csv").dropna()
Xh = ___
yh = ___
Xh_train, Xh_test, yh_train, yh_test = ___
reg = ___
ph = ___
rmse = ___
r2 = ___

assert 60000 < rmse < 80000 and 0.55 < r2 < 0.7
print(f"RMSE {rmse:,.0f} | R2 {r2:.3f}")

**Exercise 3.2** — Plot predicted vs actual values for the test set with a diagonal reference line. Where does the model struggle most?

In [ ]:
# YOUR CODE HERE

## Part 4 — Clustering with K-Means (guided practice)

### Learning Objectives
- Scale features and fit K-Means
- Choose k with the elbow method
- Interpret clusters

### Concept
K-Means repeats two steps: assign points to the nearest centre, move each centre to the mean of its points. Inertia (within-cluster squared distance) measures compactness.

### Your Turn
**Exercise 4.1** — Standardise all four Iris measurements (`Xs`), compute inertia for k = 1–8, and plot the elbow curve.

In [ ]:
iris = load_data("iris.csv")
Xs = ___
inertias = ___

assert len(inertias) == 8 and inertias[0] > inertias[-1]
plt.plot(range(1, 9), inertias, marker="o"); plt.xlabel("k"); plt.ylabel("inertia"); plt.show()

**Exercise 4.2** — Fit K-Means with k = 3 on `Xs`, add the labels to `iris` as `cluster`, and compare with `species` using `pd.crosstab`. Which species is easiest to separate?

In [ ]:
km = ___
iris["cluster"] = ___
print(pd.crosstab(iris["cluster"], iris["species"]))

## Part 5 — Deep dive: scaling and pipelines

### Concept
A `Pipeline` chains steps so `fit` learns the scaler from training data only, then the model. This prevents leakage and makes code shorter.

**Exercise 5.1** — Build `pipe` = StandardScaler + LogisticRegression on the Titanic split. Compare its accuracy with `clf`, then print the scaled coefficients sorted by size. Which feature matters most?

In [ ]:
pipe = ___
pipe_acc = ___
coefs = ___

assert pipe_acc > baseline
print(f"pipeline accuracy {pipe_acc:.3f}"); print(coefs)

## Extended practice set (Weeks 9–10)

1. **Debugging task** — the cell below has *three* bugs that make the evaluation dishonest or broken. Find and fix them.
2. **Mini-project** — Predict whether a retail transaction has a high `rating` (4–5) from `category`, `discount`, `customer_type`, `payment_method` and `customer_age`. Report accuracy against a baseline, a confusion matrix, and one honest sentence about whether the model is useful.
3. **Stretch** — Segment retail customers with K-Means on `customer_age`, `unit_price` and `units` (scaled). Name each segment.

Then take the **Module 5 quiz** and start your **Final Project**.

In [ ]:
# Bug hunt: fix this cell
scaler = StandardScaler().fit(X)                       # bug?
X_tr, X_te, y_tr, y_te = train_test_split(scaler.transform(X), y, test_size=0.2)
model = LogisticRegression(max_iter=1000).fit(X_te, y_te)   # bug?
print("accuracy:", accuracy_score(y_tr, model.predict(X_tr)))  # bug?

In [ ]:
# Mini-project: your code here